In [1]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For text processing
import re
from collections import Counter

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

# Check what’s inside your Drive root
os.listdir('/content/drive/MyDrive')

['Colab Notebooks',
 'AI ML Important Files',
 'Bank Statement',
 'Workout_Tracker.xlsx',
 'Blood Test Report ',
 'Amma Recordings',
 'Datasets']

In [4]:
os.listdir('/content/drive/MyDrive/Datasets')

['indian-job-market-dataset-2025.xlsx',
 'Copy of indian-job-market-dataset-2025.csv.xlsx']

In [5]:
file_path = '/content/drive/MyDrive/Datasets/indian-job-market-dataset-2025.xlsx'

df = pd.read_excel(file_path)

df.head()

,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Pow...",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000.0,400000.0,2.0,4.0
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162.0,4.0,"Ensure active Fire Protection System,such as F...",300000.0,500000.0,6.0,11.0
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth ...",12-18 Yrs,Not disclosed,Chennai,1324750,2892.0,4.2,MBA Marketing (preferred Tier II or III B- Sch...,0.0,0.0,12.0,18.0
3,Medical Billing Executive,270925007443,INR,6 Days Ago,GNR Global Services,"Fluent English,Spoken English,Good English Com...",0-3 Yrs,"70,000-2 Lacs PA","Mohali, Chandigarh, Kharar, Zirakpur",123804403,NaN,NaN,Job Title-Medical Billing Executive\nLocation-...,70000.0,200000.0,0.0,3.0
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephr...",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134.0,3.4,Principal Tasks & Responsibilities : (Please w...,800000.0,1800000.0,5.0,10.0


In [6]:
df.shape



(97929, 17)

In [7]:
df.columns

Index(['title', 'jobId', 'currency', 'jobUploaded', 'companyName',
       'tagsAndSkills', 'experience', 'salary', 'location', 'companyId',
       'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary',
       'maximumSalary', 'minimumExperience', 'maximumExperience'],
      dtype='object')

In [8]:
# Step 8: Clean and extract skills

# 1. Drop rows where skills are missing
df_clean = df.dropna(subset=['tagsAndSkills']).copy()

# 2. Convert to lowercase
df_clean['skills_clean'] = df_clean['tagsAndSkills'].str.lower()

# 3. Remove special characters (keep only letters, commas, spaces)
df_clean['skills_clean'] = df_clean['skills_clean'].apply(
    lambda x: re.sub(r'[^a-zA-Z, ]', '', x)
)

# 4. Split skills into list
df_clean['skills_list'] = df_clean['skills_clean'].str.split(',')

# 5. Flatten all skills into one list
all_skills = [skill.strip() for sublist in df_clean['skills_list'] for skill in sublist if skill.strip() != ""]

# 6. Count frequency
skill_counts = Counter(all_skills)

# 7. Convert to dataframe
skills_df = pd.DataFrame(skill_counts.items(), columns=['Skill', 'Count'])

# 8. Sort values
skills_df = skills_df.sort_values(by='Count', ascending=False)

# 9. Show top 20 skills
skills_df.head(20)

,Skill,Count
99,sales,9225
449,python,5860
224,project management,5553
358,c,5419
90,customer service,5131
57,sap,5059
87,management,4863
337,css,4231
361,java,4144
359,sql,3993


In [9]:
# Step 9: Standardize skills (merge duplicates and clean noise)

# Create a mapping dictionary
skill_mapping = {
    'ms excel': 'excel',
    'microsoft excel': 'excel',
    'excel ': 'excel',

    'powerbi': 'power bi',
    'power bi ': 'power bi',

    'communication skills': 'communication',
    'english': 'communication',

    'c++': 'c',

    'js': 'javascript',

    'customer service': 'customer support',

    'sales ': 'sales'
}

# Apply mapping
def clean_skill(skill):
    skill = skill.strip()
    return skill_mapping.get(skill, skill)

skills_df['Skill'] = skills_df['Skill'].apply(clean_skill)

# Group again after cleaning
skills_df = skills_df.groupby('Skill', as_index=False)['Count'].sum()

# Sort again
skills_df = skills_df.sort_values(by='Count', ascending=False)

# Remove very generic/non-analytical skills
remove_skills = [
    'management', 'development', 'communication',
    'troubleshooting', 'customer support'
]

skills_df = skills_df[~skills_df['Skill'].isin(remove_skills)]

# Show top 20 again
skills_df.head(20)

,Skill,Count
33440,sales,9225
31034,python,5860
30537,project management,5553
5365,c,5419
33858,sap,5059
9277,css,4231
20627,java,4144
36841,sql,3993
5189,business development,3758
20726,javascript,3023


In [10]:
# Step 10: Categorize skills

# Define categories
tech_skills = [
    'python', 'java', 'c', 'javascript', 'sql', 'css'
]

business_skills = [
    'sales', 'marketing', 'project management', 'business development', 'accounting'
]

tools_skills = [
    'excel', 'power bi', 'sap', 'automation'
]

# Function to assign category
def categorize(skill):
    if skill in tech_skills:
        return 'Tech'
    elif skill in business_skills:
        return 'Business'
    elif skill in tools_skills:
        return 'Tools'
    else:
        return 'Other'

# Apply category
skills_df['Category'] = skills_df['Skill'].apply(categorize)

# Aggregate counts by category
category_df = skills_df.groupby('Category')['Count'].sum().reset_index()

# Sort for clarity
category_df = category_df.sort_values(by='Count', ascending=False)

category_df

,Category,Count
1,Other,671959
2,Tech,26670
0,Business,23487
3,Tools,9568


In [11]:
# Step 11: Improved skill categorization

tech_skills = [
    'python', 'java', 'c', 'javascript', 'sql', 'css', 'html',
    'machine learning', 'data analysis', 'data science', 'deep learning',
    'artificial intelligence', 'aws', 'cloud', 'react', 'nodejs'
]

business_skills = [
    'sales', 'marketing', 'project management', 'business development',
    'accounting', 'finance', 'operations', 'strategy', 'consulting'
]

tools_skills = [
    'excel', 'power bi', 'sap', 'tableau', 'automation',
    'google analytics', 'crm', 'erp'
]

def categorize(skill):
    skill = skill.lower()
    if any(ts in skill for ts in tech_skills):
        return 'Tech'
    elif any(bs in skill for bs in business_skills):
        return 'Business'
    elif any(tl in skill for tl in tools_skills):
        return 'Tools'
    else:
        return 'Other'

skills_df['Category'] = skills_df['Skill'].apply(categorize)

category_df = skills_df.groupby('Category')['Count'].sum().reset_index()
category_df = category_df.sort_values(by='Count', ascending=False)

category_df

,Category,Count
2,Tech,371008
1,Other,290602
0,Business,46904
3,Tools,23170


In [12]:
# Step 12: Top skills within each category

# Function to get top skills per category
top_skills_by_category = {}

for category in skills_df['Category'].unique():
    top_skills = skills_df[skills_df['Category'] == category] \
                    .sort_values(by='Count', ascending=False) \
                    .head(10)

    top_skills_by_category[category] = top_skills[['Skill', 'Count']]

# Display results
for category, data in top_skills_by_category.items():
    print(f"\nTop skills in {category}:\n")
    print(data)


Top skills in Business:

                      Skill  Count
33440                 sales   9225
5189   business development   3758
23280             marketing   2509
26841            operations   2053
3883               bb sales   1540
14606           field sales   1259
11161     digital marketing   1188
38923             telesales   1065
19326          inside sales    852
33591      sales management    843

Top skills in Tech:

                    Skill  Count
31034              python   5860
30537  project management   5553
5365                    c   5419
9277                  css   4231
20627                java   4144
36841                 sql   3993
20726          javascript   3023
247            accounting   2442
1381           analytical   2170
26951              oracle   2142

Top skills in Tools:

                      Skill  Count
33858                   sap   5059
3006             automation   2276
13436                   erp   1460
3052     automation testing   1067
29269 

In [13]:
# Step 13: Better filtering for Data roles

# Filter relevant roles more precisely
data_roles = df_clean[
    df_clean['title'].str.lower().str.contains(
        'data analyst|data scientist|data engineer|analytics', regex=True
    )
]

print("Total data-related jobs:", data_roles.shape[0])

# Extract skills
skills_series = data_roles['skills_clean'].str.split(',')

all_skills_data = [
    skill.strip()
    for sublist in skills_series
    for skill in sublist
    if skill.strip() != ""
]

from collections import Counter

data_skill_counts = Counter(all_skills_data)

data_skills_df = pd.DataFrame(data_skill_counts.items(), columns=['Skill', 'Count'])
data_skills_df = data_skills_df.sort_values(by='Count', ascending=False)

data_skills_df.head(15)

Total data-related jobs: 2010


,Skill,Count
19,python,971
0,sql,614
58,scala,383
48,pyspark,379
24,data,341
28,data analysis,332
57,hive,320
23,data engineering,315
42,machine learning,254
14,hadoop,247


In [14]:
# Step 14: Compare overall vs data role skills

# Top overall skills
top_overall = skills_df.head(20)

# Top data role skills
top_data = data_skills_df.head(20)

# Convert to sets
overall_skills_set = set(top_overall['Skill'])
data_skills_set = set(top_data['Skill'])

# Skills common in both
common_skills = overall_skills_set.intersection(data_skills_set)

# Skills unique to data roles
data_unique = data_skills_set - overall_skills_set

# Skills missing in data roles (but common overall)
overall_unique = overall_skills_set - data_skills_set

print("Common Skills:\n", common_skills)
print("\nData Role Exclusive Skills:\n", data_unique)
print("\nGeneral Market Skills (Not in Data Roles):\n", overall_unique)

Common Skills:
 {'python', 'sql'}

Data Role Exclusive Skills:
 {'data', 'etl', 'snowflake', 'spark', 'power bi', 'pyspark', 'scala', 'data engineering', 'aws', 'hadoop', 'machine learning', 'gcp', 'data analytics', 'airflow', 'data analysis', 'hive', 'data modeling', 'data warehousing'}

General Market Skills (Not in Data Roles):
 {'oracle', 'c', 'javascript', 'team management', 'accounting', 'rest', 'sales', 'java', 'operations', 'business development', 'marketing', 'css', 'analytical', 'networking', 'sap', 'project management', 'agile', 'automation'}


In [15]:
# Step 14.5: Advanced categorization

def categorize_advanced(skill):
    skill = skill.lower()

    # Tech / Programming
    if any(x in skill for x in [
        'python','java','c','c++','javascript','sql','html','css',
        'react','node','angular'
    ]):
        return 'Programming'

    # Data / AI
    elif any(x in skill for x in [
        'machine learning','data','analytics','ai','artificial intelligence',
        'deep learning','nlp','computer vision'
    ]):
        return 'Data & AI'

    # Cloud / Big Data
    elif any(x in skill for x in [
        'aws','azure','gcp','hadoop','spark','hive','pyspark',
        'airflow','etl','kubernetes','docker'
    ]):
        return 'Cloud & Big Data'

    # Business / Management
    elif any(x in skill for x in [
        'sales','marketing','business','operations','strategy',
        'consulting','project management','accounting','finance'
    ]):
        return 'Business & Management'

    # Tools / Enterprise
    elif any(x in skill for x in [
        'excel','power bi','tableau','sap','erp','crm','oracle'
    ]):
        return 'Tools & Enterprise'

    # Soft Skills
    elif any(x in skill for x in [
        'communication','team','leadership','interpersonal',
        'presentation','negotiation'
    ]):
        return 'Soft Skills'

    else:
        return 'Other'

# Apply new categorization
skills_df['Better_Category'] = skills_df['Skill'].apply(categorize_advanced)

# Recalculate distribution
better_category_df = skills_df.groupby('Better_Category')['Count'].sum().reset_index()
better_category_df = better_category_df.sort_values(by='Count', ascending=False)

better_category_df

,Better_Category,Count
4,Programming,367695
3,Other,257459
0,Business & Management,49299
2,Data & AI,22661
6,Tools & Enterprise,18445
1,Cloud & Big Data,8110
5,Soft Skills,8015


In [29]:
# Install plotly (run once if needed)
!pip install plotly

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from google.colab import files

# -------------------------------
# Data
# -------------------------------
top_overall = skills_df.head(8)
top_data = data_skills_df.head(8)

gap_counts = [len(common_skills), len(data_unique), len(overall_unique)]
gap_labels = ['Common', 'Data Only', 'General Only']

# -------------------------------
# Create Dashboard WITH subplot titles
# -------------------------------
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        "Top Skills (Overall)",
        "Category Distribution",
        "Top Skills (Data Roles)",
        "Skill Gap Overview"
    ),
    specs=[[{"type": "bar"}, {"type": "pie"}],
           [{"type": "bar"}, {"type": "bar"}]],
    horizontal_spacing=0.12,
    vertical_spacing=0.18
)

# -------------------------------
# Charts
# -------------------------------
fig.add_trace(
    go.Bar(
        x=top_overall['Count'],
        y=top_overall['Skill'],
        orientation='h',
        text=top_overall['Count'],
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='#5A67D8')
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=better_category_df['Better_Category'],
        values=better_category_df['Count'],
        textinfo='percent',
        textfont=dict(color='white')
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=top_data['Count'],
        y=top_data['Skill'],
        orientation='h',
        text=top_data['Count'],
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='#14B8A6')
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=gap_labels,
        y=gap_counts,
        text=gap_counts,
        textposition='inside',
        textfont=dict(color='white'),
        marker=dict(color='#9F7AEA')
    ),
    row=2, col=2
)

# -------------------------------
# Layout (RECTANGULAR + CLEAN)
# -------------------------------
fig.update_layout(
    height=650,
    width=1200,

    title={
        'text': "Indian Job Market Skill Gap Dashboard",
        'x': 0.5,
        'font': {'size': 24, 'color': 'white'}
    },

    template="plotly_dark",
    showlegend=False,

    margin=dict(l=60, r=60, t=100, b=50),
    font=dict(color="white")
)

# -------------------------------
# Fix subplot title styling (THIS IS THE KEY FIX)
# -------------------------------
fig.update_annotations(font_size=14)

# -------------------------------
# Axis formatting
# -------------------------------
fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=2, col=1)

# -------------------------------
# Show
# -------------------------------
fig.show()

# -------------------------------
# Export (Centered + Styled)
# -------------------------------
file_name = "skill_gap_dashboard.html"

html_string = f"""
<html>
<head>
    <style>
        body {{
            margin: 0;
            background-color: #0d0d0d;
            display: flex;
            justify-content: center;
            align-items: center;
            height: 100vh;
        }}

        .dashboard {{
            padding: 20px;
            border-radius: 12px;
            background-color: #111;
            box-shadow: 0 0 25px rgba(0,0,0,0.8);
            border: 1px solid #2a2a2a;
        }}
    </style>
</head>

<body>
    <div class="dashboard">
        <div id="chart"></div>
    </div>

    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <script>
        var fig = {fig.to_json()};
        Plotly.newPlot('chart', fig.data, fig.layout);
    </script>
</body>
</html>
"""

with open(file_name, "w") as f:
    f.write(html_string)

files.download(file_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>